In [1]:
from ultralytics import YOLO
import cv2

In [2]:
from util import get_car 

In [3]:
coco_model=YOLO("yolov8n.pt")  # Load a pretrained YOLOv8n model trained on COCO dataset
license_plate_detector=YOLO("license_plate_detector.pt")  # Load a custom YOLOv8 model trained for license plate detection

In [4]:
#Load models
cap=cv2.VideoCapture("./sample.mp4")

In [5]:
vehicles=[2,3,5,7]

In [6]:
from sort.sort import *
mot_tracker=Sort()

In [ ]:
#read frames
frame_nmr=-1
ret=True
while ret:
    frame_nmr+=1
    ret,frame=cap.read()
    if ret and frame_nmr<10:
    #detect vechicles
        detections=coco_model(frame)[0]
        detections_=[]
        for detection in detections.boxes.data.tolist():
            x1,y1,x2,y2,score,class_id=detection
            print(detections)
            if int(class_id) in vehicles:
                detections_.append([x1,y2,x2,y2,score])
                
        #track vehicles
        track_ids=mot_tracker.update(np.asarray(detections_))
        
        #detect liscence plate
        license_plates=license_plate_detector(frame)[0]
        for license_plate in license_plates.boxes.data.tolist():
            x1,y1,x2,y2,score,class_id=license_plate

            #assign liscence plate to car
            xcar1,ycar1,xcar2,ycar2,car_id=get_car(license_plate,track_ids)
            
            #crop liscence plate
            liscence_plate_crop=frame[int(y1):int(y2),int(x1):int(x2), :]
            
            #process liscence plate
            liscence_plate_crop_gray=cv2.cvtColor(liscence_plate_crop,cv2.COLOR_BGR2GRAY)
            _,liscence_plate_thresh=cv2.threshold(liscence_plate_crop_gray,64,255,cv2.THRESH_BINARY_INV)
        